# Day 5 | Lab 5.3: LlamaIndex for Production-Grade RAG

**Duration:** ~1.5 hours

**Scenario.** A small AI research team needs to query ~10 internal technical research papers to answer complex multi-paper questions. We use LlamaIndex (instead of LangChain) to build a production-grade RAG pipeline — basic query engine, sub-question decomposition, multi-index routing, response synthesis modes, and a ReAct agent over query-engine tools. The corpus is **domain-agnostic technical research** (no banking) — the goal is to teach the LlamaIndex mental model cleanly.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Articulate the architectural difference between LlamaIndex and LangChain — *indices*, *query engines*, *response synthesizers* vs *retrievers*, *chains*, *runnables*.
2. Build a `VectorStoreIndex` over ChromaDB with `SimpleDirectoryReader` and `OpenAIEmbedding`.
3. Use `SubQuestionQueryEngine` to decompose a complex multi-source question.
4. Use `RouterQueryEngine` to dispatch a question to one of several specialised indices.
5. Pick the right response synthesizer (`tree_summarize` vs `compact` vs `refine`) for the task.
6. Build a `ReActAgent` whose tools are query engines.
7. Decide when to choose LlamaIndex vs LangChain for a new RAG project.

**Tools.** LlamaIndex (latest) · `llama-index-llms-anthropic` · `llama-index-embeddings-openai` · `llama-index-vector-stores-chroma` · ChromaDB · Claude Sonnet 4.5 · `text-embedding-3-small`.

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'llama-index>=0.11' 'llama-index-llms-anthropic' 'llama-index-embeddings-openai' 'llama-index-vector-stores-chroma' 'chromadb>=0.5' python-dotenv


## 2. API Key Configuration


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Imports & Global Settings

LlamaIndex uses a global `Settings` singleton for the LLM, embedding model, chunk size, and callback manager. Set it once and every index/query-engine you build inherits these defaults — this is one of the architectural choices that differs from LangChain (which threads the LLM through every chain explicitly).

In [ ]:
from llama_index.core import Settings
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = Anthropic(model='claude-sonnet-4-5-20250929', max_tokens=2048)
Settings.embed_model = OpenAIEmbedding(model='text-embedding-3-small')
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print(f'LLM:         {type(Settings.llm).__name__} · {Settings.llm.model}')
print(f'Embedding:   {type(Settings.embed_model).__name__} · {Settings.embed_model.model_name}')
print(f'Chunk size:  {Settings.chunk_size} (overlap {Settings.chunk_overlap})')


## 4. Why LlamaIndex Alongside LangChain?

LangChain and LlamaIndex are not competitors so much as **two different mental models** for the same problem space. You use LangChain for general LLM orchestration; you reach for LlamaIndex when the workload is *primarily* RAG over a complex document set.

### Conceptual mapping

| LangChain | LlamaIndex | What it represents |
|---|---|---|
| `Document` + `VectorStore` + `Retriever` | `Node` + `Index` (`VectorStoreIndex`, `SummaryIndex`, `KeywordTableIndex`, …) | *How you store and retrieve* |
| `chain = prompt \| llm \| parser` | `query_engine = index.as_query_engine(response_mode=...)` | *How you ask a question* |
| `RetrievalQA` / custom LCEL chain | `RetrieverQueryEngine` + `ResponseSynthesizer` | *Retrieve-then-synthesise* |
| `MultiQueryRetriever` | `SubQuestionQueryEngine` | *Decompose a complex query* |
| Router via `RunnableBranch` | `RouterQueryEngine` (LLM-selector or pydantic-selector) | *Dispatch to the right source* |
| `create_agent(llm, tools)` | `ReActAgent.from_tools(tools, llm)` | *Tool-using agent* |
| Callbacks → LangSmith | `Settings.callback_manager` → LlamaCloud / OpenInference | *Observability* |

### When LlamaIndex wins

- **Multi-index workloads.** You have 3 different document collections and want to route a question to the right one — `RouterQueryEngine` is purpose-built; LangChain you build it yourself.
- **Complex query decomposition.** `SubQuestionQueryEngine` does multi-hop / multi-source retrieval out of the box.
- **Heterogeneous indices.** Mix a `VectorStoreIndex` with a `SummaryIndex` and a `KeywordTableIndex` — LlamaIndex was designed for this.
- **Production RAG pipelines.** Built-in node post-processors (similarity threshold, sentence-window, metadata filters) are first-class.

### When LangChain wins

- **General LLM orchestration** — chatbots, prompt chains, multi-step pipelines that aren't only RAG.
- **Tool-heavy agents** — LangChain's tool ecosystem (LangGraph, prebuilt agents) is broader.
- **Most cohort + ecosystem material** is LangChain-first; team familiarity matters.

Most production teams end up using **both** — LlamaIndex as the RAG sub-system, wrapped behind a LangChain agent or a FastAPI service. We'll build the LlamaIndex side here.


## 5. Sample Corpus — 10 Technical Research Briefs

We synthesise a small, domain-agnostic corpus of 10 short technical briefs covering distinct topics so we can later demonstrate `RouterQueryEngine` (route between sub-corpora) and `SubQuestionQueryEngine` (decompose a multi-topic question). Each brief is written from scratch — no real-world copyrighted text.

In [ ]:
import shutil
from pathlib import Path

CORPUS_DIR = Path('./li_corpus')
if CORPUS_DIR.exists():
    shutil.rmtree(CORPUS_DIR)
CORPUS_DIR.mkdir()

# Cluster A: distributed-systems briefs (5 docs)
DOCS_A = {
    'a1_consensus.md': '''# Consensus in Distributed Systems\n\nPaxos and Raft are leader-based consensus algorithms that tolerate up to f failures with 2f+1 nodes. Raft simplifies Paxos by separating leader election, log replication, and safety. Production systems such as etcd and CockroachDB use Raft because its state machine is easier to reason about. Latency is bounded by one round-trip to a quorum of followers; throughput is bounded by the leader's ability to fan out append-entries RPCs.''',
    'a2_replication.md': '''# Data Replication Strategies\n\nSynchronous replication blocks the writer until n replicas acknowledge; asynchronous replication returns immediately and replicates in the background. Quorum-based systems (Dynamo-style) read from R replicas and write to W replicas with R + W > N to guarantee read-your-writes. Multi-leader replication tolerates partitions but introduces conflict resolution. CRDTs (counters, sets) avoid conflicts by design.''',
    'a3_partitioning.md': '''# Partitioning and Sharding\n\nRange partitioning preserves ordered scans but risks hotspots. Hash partitioning balances load but destroys locality. Consistent hashing (Dynamo, Cassandra) lets you add/remove nodes with O(K/N) data movement. Vnodes per physical node smooth the load distribution. Re-balancing strategies range from fixed-number-of-partitions (Riak, ES) to dynamic split/merge (HBase).''',
    'a4_consistency.md': '''# Consistency Models\n\nLinearizability is the strongest single-object consistency model — every operation appears to take effect atomically at some point between its invocation and response. Sequential consistency relaxes this to a single global order without real-time constraints. Causal consistency only orders operations with a happens-before relation. Eventual consistency makes no ordering guarantees, just convergence.''',
    'a5_observability.md': '''# Distributed Tracing and Observability\n\nOpenTelemetry standardises metrics, traces, and logs across language runtimes. A trace is a DAG of spans, each with a parent span ID and a context-propagated trace ID. Sampling strategies (head-based, tail-based) trade fidelity for cost. Service-level objectives (SLOs) on latency tail percentiles (p99, p99.9) drive on-call alerting.''',
}

# Cluster B: machine-learning briefs (5 docs)
DOCS_B = {
    'b1_transformers.md': '''# Transformer Architecture Notes\n\nThe transformer is built from stacked self-attention blocks plus feed-forward layers. Self-attention computes dot-product similarity between query, key, and value projections of every token, scaled by sqrt(d_k). Multi-head attention runs h parallel attention heads on lower-dimensional projections and concatenates the results. Positional encodings inject sequence order into an otherwise permutation-invariant mechanism.''',
    'b2_finetuning.md': '''# Fine-Tuning Strategies for LLMs\n\nFull fine-tuning updates every parameter and is expensive. LoRA freezes the base model and learns low-rank update matrices, slashing GPU memory by ~3x. QLoRA quantises the base model to 4-bit and trains LoRA adapters on top, fitting 65B models on a single A100. Instruction tuning aligns a base model to follow tasks; RLHF or DPO further aligns it to human preferences.''',
    'b3_evaluation.md': '''# LLM Evaluation Harnesses\n\nMMLU, ARC, HellaSwag, and BIG-bench measure broad knowledge and reasoning. Chatbot Arena measures human preference via blinded pairwise comparisons. LLM-as-Judge offers a cheaper proxy but suffers from position bias, verbosity bias, and self-enhancement bias. Domain-specific evals (legal-bench, MedQA) often correlate poorly with general benchmarks.''',
    'b4_rag_systems.md': '''# Retrieval-Augmented Generation Systems\n\nA RAG system has three knobs: a retriever (BM25, dense, hybrid, re-ranked), a chunking strategy (fixed-size, recursive, semantic, contextual), and a generator (the LLM with a retrieval-aware prompt). Quality is dominated by retrieval precision; generation quality matters less once the right context is in the prompt. RAGAS, BEIR, and MTEB are common evaluation toolkits.''',
    'b5_agents.md': '''# Agent Architectures\n\nReAct interleaves reasoning traces with tool calls. Plan-and-execute decomposes a goal into a static plan, then executes step-by-step. Reflexion adds a self-critique loop that revises plans after failures. Multi-agent systems (AutoGen, CrewAI) coordinate specialist agents — useful when role decomposition is natural, costly when it isn't.''',
}

for name, body in {**DOCS_A, **DOCS_B}.items():
    (CORPUS_DIR / name).write_text(body, encoding='utf-8')

print(f'Wrote {len(DOCS_A) + len(DOCS_B)} briefs to {CORPUS_DIR.resolve()}')
for f in sorted(CORPUS_DIR.iterdir()):
    print(f'  - {f.name}  ({f.stat().st_size} bytes)')


## 6. Loading the Corpus with `SimpleDirectoryReader`

`SimpleDirectoryReader` picks an appropriate parser per file extension (`.md`, `.pdf`, `.docx`, `.html`, …) and returns `Document` objects. Each `Document` carries `text`, `metadata`, and an auto-generated `id_`.

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(str(CORPUS_DIR)).load_data()
print(f'Loaded {len(documents)} Document(s).')
print('---')
doc0 = documents[0]
print(f'doc[0].id_       = {doc0.id_}')
print(f'doc[0].metadata  = {doc0.metadata}')
print(f'doc[0].text[:200]:\n{doc0.text[:200]}')


## 7. Build a `VectorStoreIndex` Backed by ChromaDB

`VectorStoreIndex.from_documents(...)` does the full pipeline: split documents into nodes (per `Settings.chunk_size`), embed them, and persist to the vector store. We back it with a ChromaDB collection so the index survives restarts.

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore

chroma_client = chromadb.PersistentClient(path='./chroma_li')
# Recreate collection so re-runs are deterministic.
if 'all_briefs' in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection('all_briefs')
collection = chroma_client.create_collection('all_briefs')

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True,
)
print(f'Index built. Chroma collection has {collection.count()} embedded nodes.')


## 8. Query Engine Basics — `index.as_query_engine()`

A query engine is the LlamaIndex equivalent of a LangChain RetrievalQA chain. It wraps:
1. A retriever (defaults to `VectorIndexRetriever` over the index)
2. A response synthesizer (defaults to `compact` mode)
3. Optional node post-processors (re-rankers, similarity filters)

By default it uses the global `Settings.llm`.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=4)

QUESTIONS = [
    'What does Raft simplify compared to Paxos?',
    'Why does QLoRA fit a 65B model on a single A100?',
    'What are the three knobs of a RAG system?',
]
for q in QUESTIONS:
    print(f'\nQ: {q}')
    resp = query_engine.query(q)
    print(f'A: {resp}')
    print(f'   (used {len(resp.source_nodes)} source nodes)')


## 9. Inspecting Source Nodes (Citation Surface)

Every `Response` carries `source_nodes` — the chunks that actually fed the answer. This is your citation surface; never ship a RAG answer without surfacing them.

In [ ]:
resp = query_engine.query('What is consistent hashing and why does it help when nodes change?')
print(f'Answer: {resp}\n')
print('Source nodes (top {}):'.format(len(resp.source_nodes)))
for i, sn in enumerate(resp.source_nodes, 1):
    fn = sn.node.metadata.get('file_name', '?')
    print(f'  [{i}] file={fn}  score={sn.score:.4f}')
    print(f'      text[:120] = {sn.node.text[:120]!r}')


## 10. Sub-Question Query Engine — Decomposing Complex Questions

A complex question often spans multiple sub-topics that no single chunk answers. `SubQuestionQueryEngine` uses the LLM to break the question into sub-questions, queries each against an underlying engine, and synthesises a single answer.

**When it wins:** *"Compare X and Y"* type questions where X and Y live in different documents.

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine

# Wrap our basic engine as a single tool the sub-question engine can route to.
all_briefs_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name='all_briefs',
        description='Technical briefs covering distributed systems and machine-learning topics.',
    ),
)

sub_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[all_briefs_tool],
    verbose=True,
)

complex_q = (
    'Compare how distributed-systems Raft uses consensus to tolerate failures '
    'and how transformer attention computes similarity between tokens. '
    'What does each take as input, and what guarantee does each provide?'
)
resp = sub_engine.query(complex_q)
print('\n=== Final synthesised answer ===')
print(resp)


## 11. Router Query Engine — Dispatch Between Multiple Indices

Real workloads often have **multiple distinct corpora** (research papers, internal wiki, customer tickets). `RouterQueryEngine` uses the LLM (or a Pydantic schema) to select the right tool per query — cheaper and more accurate than indexing everything into one giant store.

We build two specialised indices — one for distributed systems (cluster A), one for ML (cluster B) — and let the router pick.

In [ ]:
# Filter documents by source file into two specialist indices.
docs_a = [d for d in documents if d.metadata.get('file_name', '').startswith('a')]
docs_b = [d for d in documents if d.metadata.get('file_name', '').startswith('b')]

for name in ['ds_briefs', 'ml_briefs']:
    if name in [c.name for c in chroma_client.list_collections()]:
        chroma_client.delete_collection(name)

ds_collection = chroma_client.create_collection('ds_briefs')
ml_collection = chroma_client.create_collection('ml_briefs')

ds_index = VectorStoreIndex.from_documents(
    docs_a,
    storage_context=StorageContext.from_defaults(
        vector_store=ChromaVectorStore(chroma_collection=ds_collection)
    ),
)
ml_index = VectorStoreIndex.from_documents(
    docs_b,
    storage_context=StorageContext.from_defaults(
        vector_store=ChromaVectorStore(chroma_collection=ml_collection)
    ),
)
print(f'Distributed-systems index: {ds_collection.count()} nodes')
print(f'Machine-learning index:    {ml_collection.count()} nodes')


In [ ]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

ds_tool = QueryEngineTool(
    query_engine=ds_index.as_query_engine(similarity_top_k=3),
    metadata=ToolMetadata(
        name='distributed_systems',
        description='Briefs about consensus (Paxos, Raft), replication, partitioning, '
                    'consistency models, and distributed observability.',
    ),
)
ml_tool = QueryEngineTool(
    query_engine=ml_index.as_query_engine(similarity_top_k=3),
    metadata=ToolMetadata(
        name='machine_learning',
        description='Briefs about transformer architecture, fine-tuning (LoRA/QLoRA), '
                    'LLM evaluation, RAG systems, and agent architectures.',
    ),
)

router = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[ds_tool, ml_tool],
    verbose=True,
)

for q in [
    'What does the lambda parameter control in MMR?  (trick — neither index has this)',
    'Explain the multi-head attention mechanism.',
    'How does consistent hashing reduce data movement when adding nodes?',
]:
    print(f'\nQ: {q}')
    print(f'A: {router.query(q)}')


## 12. Response Synthesizers — `tree_summarize` vs `compact` vs `refine`

Once retrieval returns N relevant chunks, the **response synthesizer** decides how to feed them to the LLM:

| Mode | How it works | Best for | Cost |
|---|---|---|---|
| `compact` *(default)* | Pack as many chunks into one prompt as the context window allows; send once. | Most queries — fast, cheap. | 1 LLM call |
| `refine` | Send the question + chunk 1, get an answer. Then for each remaining chunk, send the running answer + the chunk and *refine* the answer. | Long-form synthesis where order matters. | N LLM calls |
| `tree_summarize` | Group chunks into pairs/triples, summarise each group, then summarise the summaries — recursively. | Summarisation of *many* documents. | O(N/k) LLM calls per level |

Default to `compact`; switch to `tree_summarize` for summarisation-heavy queries; reach for `refine` when you need a long, coherent narrative built up chunk by chunk.

In [ ]:
from llama_index.core.response_synthesizers import get_response_synthesizer

QUESTION = 'Summarise everything you know about consistency and consensus from these briefs.'
modes = ['compact', 'refine', 'tree_summarize']
for mode in modes:
    synthesizer = get_response_synthesizer(response_mode=mode)
    qe = ds_index.as_query_engine(similarity_top_k=5, response_synthesizer=synthesizer)
    resp = qe.query(QUESTION)
    text = str(resp)
    print(f'\n--- mode={mode} | {len(text)} chars | nodes={len(resp.source_nodes)} ---')
    print(text[:380] + ('…' if len(text) > 380 else ''))


## 13. LlamaIndex `ReActAgent` — Agent over Query-Engine Tools

`ReActAgent` is LlamaIndex's tool-using agent. The crucial difference from LangChain's `create_agent` is that **the natural unit of a LlamaIndex tool is a query engine**, not a generic function. You expose your indices as tools; the agent decides which to query, possibly multiple times in a loop, before synthesising an answer.

In [ ]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent.from_tools(
    tools=[ds_tool, ml_tool],
    llm=Settings.llm,
    verbose=True,
    max_iterations=8,
)

complex_q = (
    'I am building a RAG system that needs to scale across multiple data-centres. '
    'Which consensus algorithm should I look at for the metadata store, and which RAG '
    'evaluation toolkit should I use to benchmark retrieval quality?'
)
resp = agent.chat(complex_q)
print('\n=== Agent answer ===')
print(resp)


## 14. LlamaIndex vs LangChain — Decision Matrix

| Situation | LlamaIndex | LangChain |
|---|---|---|
| Single document collection, simple Q&A | ✅ trivial (`as_query_engine`) | ✅ trivial (LCEL chain) |
| Multiple heterogeneous indices needing routing | ✅ `RouterQueryEngine` built-in | ⚠️ build with `RunnableBranch` |
| Multi-hop / decomposed questions | ✅ `SubQuestionQueryEngine` built-in | ⚠️ build with `MultiQueryRetriever` + custom logic |
| Tool-using agent over arbitrary functions | ⚠️ possible, less ergonomic | ✅ `create_agent` is native |
| Streaming token-by-token to a UI | ✅ `as_query_engine(streaming=True)` | ✅ `.astream()` / `.astream_events()` |
| Production observability (LangSmith, LangFuse) | ⚠️ via OpenInference adapter | ✅ first-class |
| Complex prompt engineering / orchestration outside RAG | ❌ wrong tool | ✅ home turf |
| Many specialised retrievers (BM25, hybrid, contextual) on the same store | ⚠️ less composable | ✅ `EnsembleRetriever`, `ContextualCompressionRetriever` |

**Practical rule.** If RAG is the centre of gravity of your app, start with LlamaIndex and wrap it in a thin LangChain/FastAPI layer. If your app is conversational + tool-heavy with RAG as one of many sub-systems, stay in LangChain and import LlamaIndex query engines as tools where you need their power.

## 15. Production Deployment Notes

- **Persist your index.** `ChromaVectorStore` is durable on disk; for a managed vector DB, swap in `PineconeVectorStore`, `WeaviateVectorStore`, `QdrantVectorStore`, etc. The query-engine code above is unchanged.
- **LlamaCloud** is LlamaIndex's hosted parsing + indexing service — useful when your corpus is PDFs/scans/tables that `SimpleDirectoryReader` mangles. It's the *parsing* tier you'd plug into the same `VectorStoreIndex` API.
- **Caching.** Wrap `Settings.llm` and `Settings.embed_model` with caching middleware in dev to avoid re-paying for identical embeddings/completions.
- **Observability.** Configure `Settings.callback_manager` with the OpenInference handler — this exports spans to Phoenix / Langfuse / OpenTelemetry collectors.
- **Cost.** Most cost is the *embedding* pass at index time, not query time. Re-embed only when documents change; persist the Chroma directory between runs.

---
## 16. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **LlamaIndex mental model** | *Indices* hold knowledge; *query engines* answer questions; *tools* expose them to agents. Different shape from LangChain's runnable-composition model. |
| **`SimpleDirectoryReader`** | One-line corpus ingest; auto-picks parser per extension. |
| **`VectorStoreIndex` on Chroma** | Persistent, swap to Pinecone/Weaviate/Qdrant by changing one import. |
| **`SubQuestionQueryEngine`** | Decompose 'compare X and Y' style questions automatically. |
| **`RouterQueryEngine`** | Dispatch a question to the right specialist index — purpose-built, hard to beat with a hand-rolled `RunnableBranch`. |
| **Synthesizer modes** | Default `compact`. `tree_summarize` for many-doc summarisation. `refine` for long coherent narratives. |
| **`ReActAgent`** | Tool-using agent whose tools are query engines; differs from LangChain's `create_agent` in that the *natural* tool is a query engine, not an arbitrary function. |
| **LlamaIndex vs LangChain** | Pick by centre of gravity. Most production stacks use both. |

**Next Lab:** Lab 5.4 — RAG Evaluation: RAGAS + Custom Retrieval Metrics


## 17. Stretch Exercise (Optional)

1. Add a third index — a `SummaryIndex` over the same documents — and add it as a third tool to the `RouterQueryEngine`. Test with summarisation queries vs factual queries and observe the routing.
2. Replace `LLMSingleSelector` with `LLMMultiSelector` and watch the router fan out to multiple tools for a multi-aspect question.
3. Wrap `query_engine` with a `SimilarityPostprocessor(similarity_cutoff=0.75)` and re-run the basic queries — observe which questions now refuse to answer (no relevant context).
4. Convert one of the LangChain RAG chains from Lab 4.3 to LlamaIndex and time both. Note the LOC delta and where the trade-offs land.
5. Persist the Chroma collection, restart the kernel, and re-load the existing index with `VectorStoreIndex.from_vector_store(...)` instead of re-embedding. Confirm query latency is unchanged.


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. LlamaIndex vs LangChain for RAG — what's the architectural difference?**

*Hint:* Think *unit of composition*: indices vs runnables.

*Answer sketch:* LlamaIndex is built around indices and query engines (knowledge → questions). LangChain is built around runnables composed with `|` (function composition). LlamaIndex ships RAG-specific abstractions (Sub-Question, Router, multiple synthesizers) as first-class; LangChain ships a general orchestration kit and you assemble RAG from runnables and retrievers.

---

**Q2. What is a Sub-Question Query Engine — when does it beat a single-pass RAG?**

*Hint:* Think 'compare X and Y' across documents.

*Answer sketch:* It uses the LLM to decompose a complex question into independent sub-questions, queries each against an underlying engine, then synthesises one answer. It beats single-pass RAG when the answer requires combining information that lives in *different* chunks/documents and no single chunk contains both — classic comparison or multi-hop questions.

---

**Q3. Tree-summarize vs compact vs refine — what's the trade-off?**

*Hint:* Number of LLM calls vs context-window pressure vs synthesis style.

*Answer sketch:* `compact` packs as many nodes as possible into one prompt → 1 LLM call, cheapest, fine for most queries. `refine` makes N sequential calls (answer with chunk 1, then refine with chunk 2, …) → expensive but produces coherent long-form output. `tree_summarize` builds a tree of summaries → O(N/k) calls per level; best when you need to summarise *many* documents and they don't all fit in context.

---

**Q4. When use a Router Query Engine over a multi-source retriever?**

*Hint:* Think *cost* and *precision* of routing vs *recall*.

*Answer sketch:* Use a router when your corpora are mechanically distinct (different schemas, different topics, different update cadences) and most queries belong to *one* corpus — routing is cheaper and more precise than embedding-everything-together. Use a multi-source retriever when queries genuinely span sources and you'd rather increase recall than save a few embeddings.

---

**Q5. How does LlamaIndex's `ReActAgent` differ from LangChain's `create_agent`?**

*Hint:* Think *tool granularity* and *prebuilt RAG ergonomics*.

*Answer sketch:* Both implement a tool-calling loop. The difference is idiomatic: in LlamaIndex the natural tool is a `QueryEngineTool` (a query engine wrapped as a tool) — you expose RAG sub-systems and the agent picks. LangChain's tools are arbitrary functions — broader, but you assemble RAG behind each tool yourself. LangChain's agent ecosystem (LangGraph, MCP, prebuilt agents) is also wider.

---

**Q6. What's the equivalent of LangChain's LCEL in LlamaIndex?**

*Hint:* Trick question — there isn't a 1:1 equivalent.

*Answer sketch:* There's no `|`-pipe DSL in LlamaIndex. You compose by *configuring* a query engine — pick a retriever, pick a response synthesizer, pick post-processors — rather than chaining runnables. For complex flows LlamaIndex offers `QueryPipeline` (DAG of modules) which is the closest analogue, but it's a smaller surface than LCEL.

---

**Q7. Why does LlamaIndex have a separate 'Index' abstraction — what does it buy?**

*Hint:* Multiple indexing strategies on the same documents.

*Answer sketch:* Different question types want different indexing strategies: vector for semantic, summary for high-level Q&A, keyword-table for exact-match recall, knowledge-graph for entity walks. LlamaIndex makes Index a first-class abstraction so you can build several over the same documents, expose each as a tool, and route. LangChain treats this as 'just a vector store + retriever' which makes the heterogeneous case awkward.

---

**Q8. How would you migrate a LangChain RAG pipeline to LlamaIndex (and why might you not)?**

*Hint:* Map retrievers → indices, chains → query engines; weigh team familiarity.

*Answer sketch:* Mechanically: replace `vectorstore.as_retriever()` with `index.as_query_engine()`, replace your hand-rolled `format_docs | prompt | llm | parser` chain with a response synthesizer mode, expose extra sources via `RouterQueryEngine`. You might NOT migrate when the team is LangChain-native, when LangSmith observability is critical, or when RAG is only one of many sub-systems — the cognitive cost of a second framework usually outweighs LlamaIndex's RAG-specific wins.

